# Frontier RM medallion build
Loads the versioned synthetic landing files into schema-enabled Bronze, Silver, and Gold Delta tables.

In [ ]:
from pyspark.sql import functions as F

snapshot_at = '2026-08-12T08:30:00+08:00'
landing_path = 'Files/bronze/landing/seed=20260812'
datasets = [
    'relationship_managers', 'customers', 'households', 'accounts',
    'holdings', 'products', 'transactions', 'interactions',
    'compliance_profiles', 'opportunities', 'customer_events',
    'market_snapshots', 'rm_actions', 'client_advisory_profiles',
    'risk_profile_history', 'client_investment_activity',
    'observed_behaviour_history', 'cio_houseview_reports',
    'cio_houseview_sections', 'regulatory_documents', 'regulatory_rules',
]

for schema_name in ('bronze', 'silver', 'gold'):
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema_name}')

row_counts = {}
for dataset in datasets:
    source = spark.read.json(f'{landing_path}/{dataset}.jsonl')
    bronze = (source
        .withColumn('_ingested_at', F.to_timestamp(F.lit(snapshot_at)))
        .withColumn('_source_file', F.lit(f'{dataset}.jsonl'))
        .withColumn('_batch_id', F.lit('seed-20260812')))
    bronze.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'bronze.{dataset}')
    silver = source.dropDuplicates()
    silver.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'silver.{dataset}')
    row_counts[dataset] = silver.count()

customers = spark.table('silver.customers').alias('c')
households = spark.table('silver.households').alias('h')
compliance = spark.table('silver.compliance_profiles').alias('cp')
accounts = spark.table('silver.accounts').alias('a')
holdings = spark.table('silver.holdings').alias('hd')
products = spark.table('silver.products').alias('p')
events = spark.table('silver.customer_events').alias('e')
interactions = spark.table('silver.interactions').alias('i')
opportunities = spark.table('silver.opportunities').alias('o')
actions = spark.table('silver.rm_actions').alias('ra')
advisory_profiles = spark.table('silver.client_advisory_profiles').alias('ap')
activities = spark.table('silver.client_investment_activity').alias('ia')
behaviour = spark.table('silver.observed_behaviour_history').alias('bh')
houseview_reports = spark.table('silver.cio_houseview_reports').alias('hr')
houseview_sections = spark.table('silver.cio_houseview_sections').alias('hs')
regulatory_documents = spark.table('silver.regulatory_documents').alias('rd')
regulatory_rules = spark.table('silver.regulatory_rules').alias('rr')

customer_360 = (customers
    .join(households, F.col('c.household_id') == F.col('h.household_id'), 'left')
    .join(compliance, F.col('c.customer_id') == F.col('cp.customer_id'), 'left')
    .select(
        F.col('c.customer_id'), F.col('c.rm_id'), F.col('c.full_name'), F.col('c.segment'),
        F.col('h.household_name'), F.col('c.relationship_value'), F.col('c.currency'),
        F.col('c.risk_profile'), F.col('c.contact_preference'), F.col('c.consent_status'),
        F.col('cp.kyc_status'), F.col('cp.kyc_due_date'), F.col('cp.suitability_status'),
    ))
customer_360.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold.customer_360')

portfolio_exposure = (holdings
    .join(accounts, F.col('hd.account_id') == F.col('a.account_id'))
    .join(products, F.col('hd.product_id') == F.col('p.product_id'))
    .groupBy(F.col('a.customer_id'), F.col('p.asset_class'), F.col('hd.currency'))
    .agg(F.sum('hd.market_value').alias('market_value')))
portfolio_exposure.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold.portfolio_exposure')

maturity_watchlist = (events
    .filter(F.col('e.event_type') == 'FIXED_DEPOSIT_MATURITY')
    .join(customers, F.col('e.customer_id') == F.col('c.customer_id'))
    .select(F.col('e.event_id'), F.col('c.customer_id'), F.col('c.full_name'), F.col('e.event_value'), F.col('e.currency'), F.col('e.maturity_date'), F.col('c.rm_id')))
maturity_watchlist.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold.maturity_watchlist')

engagement_gap = (customers
    .join(interactions, F.col('c.customer_id') == F.col('i.customer_id'), 'left')
    .select(F.col('c.customer_id'), F.col('c.full_name'), F.col('i.occurred_at').alias('last_interaction_at'), F.col('i.channel'), F.col('i.summary'), F.col('c.rm_id')))
engagement_gap.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold.engagement_gap')

compliance_due = (customers
    .join(compliance, F.col('c.customer_id') == F.col('cp.customer_id'))
    .filter((F.col('cp.kyc_status') != 'CURRENT') | (F.col('cp.suitability_status') != 'CURRENT'))
    .select(F.col('c.customer_id'), F.col('c.full_name'), F.col('cp.kyc_status'), F.col('cp.kyc_due_date'), F.col('cp.suitability_status'), F.col('c.rm_id')))
compliance_due.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold.compliance_due')

rm_opportunity_snapshot = (opportunities
    .join(customers, F.col('o.customer_id') == F.col('c.customer_id'))
    .select(F.col('o.opportunity_id'), F.col('o.event_id'), F.col('c.customer_id'), F.col('c.full_name'), F.col('o.title'), F.col('o.opportunity_type'), F.col('o.priority'), F.col('o.status'), F.col('o.estimated_value'), F.col('o.currency'), F.col('o.confidence_score'), F.col('o.rm_id')))
rm_opportunity_snapshot.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold.rm_opportunity_snapshot')

event_summary = (events.groupBy('customer_id')
    .agg(F.concat_ws(' | ', F.sort_array(F.collect_set('event_type'))).alias('event_types')))
meeting_context = (customer_360.alias('g')
    .join(rm_opportunity_snapshot.alias('os'), F.col('g.customer_id') == F.col('os.customer_id'), 'left')
    .join(event_summary.alias('es'), F.col('g.customer_id') == F.col('es.customer_id'), 'left')
    .join(actions, F.col('g.customer_id') == F.col('ra.customer_id'), 'left')
    .select(
        F.col('g.customer_id'), F.col('g.full_name'), F.col('g.household_name'),
        F.col('g.relationship_value'), F.col('g.currency'), F.col('g.risk_profile'),
        F.col('g.contact_preference'), F.col('g.consent_status'), F.col('g.kyc_status'),
        F.col('os.opportunity_id'), F.col('os.event_id'), F.col('os.title').alias('opportunity_title'),
        F.col('os.priority'), F.col('os.confidence_score'), F.col('es.event_types'),
        F.col('ra.action_type'), F.col('ra.due_at'), F.col('g.rm_id'),
    ))
meeting_context.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold.meeting_context')

client_advisory_context = advisory_profiles.select(
    'customer_id', 'employment_status', 'retirement_status',
    'declared_risk_score', 'declared_risk_label',
    'observed_behaviour_indicator', 'observed_behaviour_label',
    'risk_review_status', 'profile_effective_at', 'profile_review_due_at',
    'indicator_calculated_at', 'liquidity_horizon_months',
    'income_complete', 'commitments_complete', 'knowledge_experience_status',
    'cka_status', 'car_status', 'selected_client_status',
)
client_advisory_context.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold.client_advisory_context')

client_activity_evidence = (activities
    .join(behaviour, F.col('ia.investment_activity_id') == F.col('bh.trigger_activity_id'), 'left')
    .select(
        F.col('ia.investment_activity_id'), F.col('ia.customer_id'), F.col('ia.account_id'),
        F.col('ia.activity_type'), F.col('ia.asset_class'), F.col('ia.product_id'),
        F.col('ia.amount'), F.col('ia.quantity'), F.col('ia.currency'), F.col('ia.activity_at'),
        F.col('ia.source'), F.col('ia.explanation'), F.col('bh.previous_indicator'),
        F.col('bh.observed_indicator'), F.col('bh.declared_profile_changed'),
        F.col('bh.risk_review_status'), F.col('bh.calculated_at'),
    ))
client_activity_evidence.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold.client_activity_evidence')

houseview_document_index = (houseview_reports
    .join(houseview_sections, F.col('hr.houseview_id') == F.col('hs.houseview_id'))
    .select(
        F.col('hr.houseview_id'), F.col('hs.houseview_section_id'),
        F.col('hr.title').alias('report_title'), F.col('hs.title').alias('section_title'),
        F.col('hr.as_of_date'), F.col('hr.status'), F.col('hr.cio_stance'),
        F.col('hr.executive_summary'), F.col('hr.document_path'),
        F.col('hs.sequence'), F.col('hs.view'), F.col('hs.positioning'), F.col('hs.risks'),
    ))
houseview_document_index.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold.houseview_document_index')

regulatory_control_register = (regulatory_rules
    .join(regulatory_documents, F.col('rr.regulatory_document_id') == F.col('rd.regulatory_document_id'))
    .select(
        F.col('rr.regulatory_rule_id'), F.col('rr.regulatory_document_id'),
        F.col('rr.paragraph'), F.col('rr.title'), F.col('rr.summary'),
        F.col('rr.applies_to'), F.col('rr.gate'), F.col('rd.source_title'),
        F.col('rd.source_last_updated'), F.col('rd.document_path'), F.col('rd.disclaimer'),
    ))
regulatory_control_register.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold.regulatory_control_register')

active_houseview = houseview_reports.filter(F.col('hr.status') == 'ACTIVE').select('houseview_id')
recommendation_grounding_context = (customer_360.alias('g')
    .join(client_advisory_context.alias('ac'), F.col('g.customer_id') == F.col('ac.customer_id'))
    .join(client_activity_evidence.alias('ae'), F.col('g.customer_id') == F.col('ae.customer_id'), 'left')
    .join(rm_opportunity_snapshot.alias('os'), F.col('g.customer_id') == F.col('os.customer_id'), 'left')
    .crossJoin(active_houseview.alias('ah'))
    .select(
        F.col('g.customer_id'), F.col('g.full_name'), F.col('os.opportunity_id'),
        F.col('ah.houseview_id'), F.col('ac.declared_risk_score'),
        F.col('ac.observed_behaviour_indicator'), F.col('ac.risk_review_status'),
        F.col('ac.employment_status'), F.col('ae.investment_activity_id'),
        F.col('ae.activity_type'), F.col('ae.asset_class'), F.col('ae.activity_at'),
    ))
recommendation_grounding_context.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold.recommendation_grounding_context')

gold_tables = [
    'customer_360', 'portfolio_exposure', 'maturity_watchlist', 'engagement_gap',
    'compliance_due', 'rm_opportunity_snapshot', 'meeting_context',
    'client_advisory_context', 'client_activity_evidence', 'houseview_document_index',
    'regulatory_control_register', 'recommendation_grounding_context',
]
validation = {name: spark.table(f'gold.{name}').count() for name in gold_tables}
assert row_counts['customers'] == 20
assert row_counts['cio_houseview_reports'] == 2
assert spark.table('gold.meeting_context').filter(F.col('customer_id') == 'client-lim').count() == 1
assert spark.table('gold.compliance_due').filter(F.col('customer_id') == 'client-tan').count() == 1
assert spark.table('gold.client_advisory_context').filter(
    (F.col('customer_id') == 'client-lim') &
    (F.col('declared_risk_score') == 3) &
    (F.col('observed_behaviour_indicator') == 2)
).count() == 1
assert spark.table('gold.client_activity_evidence').filter(
    (F.col('customer_id') == 'client-lim') &
    (F.col('activity_type') == 'SELL') &
    (F.col('declared_profile_changed') == F.lit(False))
).count() == 1
assert spark.table('gold.client_advisory_context').filter(
    (F.col('customer_id') == 'client-tan') &
    (F.col('retirement_status') == 'RETIRED') &
    (F.col('risk_review_status') == 'REVIEW_REQUIRED')
).count() == 1
assert spark.table('gold.houseview_document_index').filter(F.col('status') == 'ACTIVE').select('houseview_id').distinct().count() == 1
print({'silverRowCounts': row_counts, 'goldRowCounts': validation, 'snapshotAt': snapshot_at})